# 粒径分布 Canonical Workflow

独立 PSD workflow canonical，覆盖旧 `粒径分布.ipynb` 与 `PSD_to_COMSOL.ipynb` 的能力并集。

## Feature Parity

| 旧能力 | canonical 保留方式 |
|---|---|
| 多材料 PSD 输入 | 顶部 `CONFIG["materials"]` 或 `DatasetQuery` 注册数据 |
| PSD 拟合 | single/bimodal lognormal 均保留 |
| 材料对比 | `material_summary` 与可选 `case_summary` |
| 仿真 | `run_mode=study` 启用 single vs PSD PyBaMM 对比 |
| 电压曲线 | `step_curves` / `full_curves` 输出和图表 |
| 离散与 COMSOL 转换 | Dv 百分位与 raw PSD 均可生成 histogram |
| 导出 | 标准 `BatteryProject/output/runs/psd/<run_id>/` |


In [ ]:
CONFIG = {
    "cell": "MIC",
    "run_mode": "smoke",
    "output_name": "粒径分布",
    "selected_strategy": "bimodal",
    "keep_percent": 99.0,
    "default_diameters_um": [0.243, 0.276, 0.314, 0.357, 0.405, 0.460, 0.523, 0.594, 0.675],
    "materials": {
        "B": {"Dv50_um": 0.870, "vol_pct": [0.50, 1.75, 3.46, 5.16, 6.41, 7.03, 7.05, 6.69, 6.22]},
        "C": {"Dv50_um": 1.288, "vol_pct": [0.00, 0.34, 1.20, 2.40, 3.61, 4.56, 5.14, 5.38, 5.39]},
    },
    "datasets": [],
    "temperature_k": 298.15,
    "nominal_capacity_ah": 314.0,
    "nominal_voltage_v": 3.2,
    "charge_cutoff_v": 3.65,
    "discharge_cutoff_v": 2.5,
    "period_minutes": 0.5,
    "rest_minutes": 30.0,
    "cycles": 1,
    "initial_soc": 0.5,
    "negative_sd_rel": 0.3,
    "rate_list": [0.5],
    "power_w_list": None,
    "summary_cycle": 1,
    "parameter_overrides": {},
    "modes": {
        "smoke": {"run_simulation": False, "run_comsol_conversion": True, "cycles": 1},
        "study": {"run_simulation": True, "run_comsol_conversion": True, "cycles": 2},
    },
    "comsol_dv": [
        {
            "name": "LFP",
            "dv_percentiles_um": {10: 0.432, 50: 1.262, 90: 3.843, 99: 6.742},
            "n_bins": 11,
            "rp_min_um": 0.1,
            "rp_max_um": 3.5,
            "total_count": 100,
            "weighting": "surface",
            "function_tag": "int1",
        }
    ],
    "comsol_raw": [
        {
            "name": "YA15",
            "diameters_um": [0.243, 0.276, 0.314, 0.357, 0.405, 0.460, 0.523, 0.594, 0.675],
            "vol_pct": [0.51, 1.74, 3.43, 5.09, 6.28, 6.81, 6.70, 6.19, 5.56],
            "n_bins": 10,
            "total_count": 100,
            "function_tag": "int2",
            "function_name": "f_histYA15",
        }
    ],
}


## 配置校验与数据查询

数据入口通过 `DatasetQuery` 表达；`CONFIG["datasets"]` 非空时会从 `datasets.json` resolve。

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
plt.style.use("science")
plt.rcParams["font.family"] = "Calibri, Microsoft YaHei"
import pandas as pd

SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in (SEARCH_ROOT, *SEARCH_ROOT.parents):
    if (candidate / "src" / "easy_imports.py").exists() and (candidate / "pyproject.toml").exists():
        PROJECT_ROOT = candidate
        break
    nested = candidate / "BatteryProject"
    if (nested / "src" / "easy_imports.py").exists() and (nested / "pyproject.toml").exists():
        PROJECT_ROOT = nested
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Cannot locate BatteryProject root from {SEARCH_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook import setup_notebook
from src.workflow_specs import DatasetQuery
from src.workflows import psd as psd_workflow
from src.psd_workflow import evaluate_bimodal_fit, evaluate_single_fit

ctx = setup_notebook(cell=CONFIG["cell"], project_root=PROJECT_ROOT, style="science", autoreload=True)
importlib.reload(psd_workflow)
PsdWorkflowSpec = psd_workflow.PsdWorkflowSpec
run_psd_workflow = psd_workflow.run_psd_workflow
spec = PsdWorkflowSpec.from_mapping(CONFIG)
print(f"project_root = {ctx.project_root}")
print(f"DatasetQuery count = {len(spec.datasets)}")
spec


## 运行 Headless PSD Workflow

标准输出写入 `BatteryProject/output/runs/psd/<run_id>/`。

In [ ]:
result = run_psd_workflow(spec, project_root=ctx.project_root, workspace_root=ctx.workspace_root)
print(result["context"].run_dir)
display(result["metrics"])
display(result["material_summary"])


## PSD 拟合

展示体积分布、面积加权分布、single lognormal 和 bimodal lognormal 拟合。

In [ ]:
material_analysis = result["material_analysis"]
color_map = dict(zip(material_analysis.keys(), plt.cm.tab10.colors))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for material_name, material in material_analysis.items():
    color = color_map[material_name]
    distribution = material["distribution"]
    radii_um = distribution["radii_m"] * 1e6
    axes[0].semilogx(distribution["diameters_um"], distribution["vol_pct"], marker="o", ms=3, label=material_name, color=color)
    axes[1].plot(radii_um, distribution["area_pdf_um_inv"], color=color, lw=2, alpha=0.5, label=f"{material_name} measured")
    axes[1].plot(radii_um, evaluate_single_fit(distribution["radii_m"], material["single_fit"]) * 1e-6, color=color, lw=1.2, ls=":", label=f"{material_name} single")
    axes[1].plot(radii_um, evaluate_bimodal_fit(distribution["radii_m"], material["bimodal_fit"]) * 1e-6, color=color, lw=1.2, ls="--", label=f"{material_name} bimodal")
axes[0].set_xlabel("Particle diameter [um]")
axes[0].set_ylabel("Volume distribution [%]")
axes[0].set_title("Input volume distributions")
axes[0].grid(True, alpha=0.3)
axes[0].legend(fontsize=8)
axes[1].set_xlabel("Particle radius [um]")
axes[1].set_ylabel("Area-weighted density [um^-1]")
axes[1].set_title("Measured vs fitted PSD")
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## 材料对比

`material_summary` 保留 PSD 摘要；启用仿真后 `case_summary` 给出 single/PSD 能效和容量对比排序。

In [ ]:
display(result["material_summary"].sort_values("area_mean_radius_um").reset_index(drop=True))
if result["case_summary"].empty:
    print("当前 run_mode 未启用仿真；切换 CONFIG['run_mode']='study' 后生成 case_summary。")
else:
    display(result["case_summary"].sort_values(["case_label", "psd_efficiency_pct"], ascending=[True, False]))


## 仿真

`study` 模式调用独立 PSD runner 的 single-particle vs particle-size-distribution PyBaMM 对比；长工况不在 canonical 默认执行。

In [ ]:
if result["study"] is None:
    print("仿真未执行。将 CONFIG['run_mode'] 改为 'study' 后再运行本 Notebook。")
else:
    display(result["case_summary"])


## 电压曲线

启用仿真后展示 charge/discharge step curves 与 full voltage-time curves。

In [ ]:
step_curves = result["step_curves"]
if step_curves.empty:
    print("没有 step_curves；当前 run_mode 未执行仿真。")
else:
    display(step_curves.head())


## 离散与 COMSOL 转换

保留 Dv 百分位拟合、number/surface/volume 加权、raw PSD 离散化和 Java API 片段。

In [ ]:
comsol_histograms = result["comsol_histograms"]
display(comsol_histograms)
for name, snippet in result["java_snippets"].items():
    print("=" * 80)
    print(name)
    print(snippet)


## 导出

所有 artifact 已由 runner 写入标准 run 目录；此处只展示路径。

In [ ]:
artifact_paths = pd.DataFrame([{"artifact": name, "path": str(path)} for name, path in result["artifact_paths"].items()])
display(artifact_paths)
print(f"metrics: {result['metrics_path']}")
print(f"workbook: {result['summary_workbook_path']}")
